In [17]:
from langgraph.graph import StateGraph , START , END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict ,Literal , Annotated

In [18]:
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [19]:
class Tweet(TypedDict):

    topic: str
    tweet: str
    evaluation: Literal['approved','need_improvement']
    feedback: str
    iteration: int
    max_iteration : int
    

In [20]:
def generate_tweet(state: Tweet) -> Tweet:
    prompt =f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
"""

    generated_tweet = model.invoke(prompt)    
    return {'tweet': generated_tweet}

In [21]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [23]:
structured_output = model.with_structured_output(TweetEvaluation)

In [22]:
def evaluate_tweet(state: Tweet) -> Tweet:
    prompt = f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
"""
    
    evaluation_result = structured_output.invoke(prompt)


    return {'evaluation': evaluation_result.evaluation, 'feedback': evaluation_result.feedback}

In [24]:
def optimize_tweet(state: Tweet) -> Tweet:
    prompt = f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
"""
    response = model.invoke(prompt).text
    iteration = state['iteration'] + 1
    return {'tweet': response, 'iteration': iteration}

In [25]:
def route(state: Tweet) -> str:
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'need_improvement'

In [26]:
graph  = StateGraph(Tweet)

graph.add_node('generate' , generate_tweet)
graph.add_node('evaluate' , evaluate_tweet)
graph.add_node('optimize' , optimize_tweet)

graph.add_edge(START , 'generate')
graph.add_edge('generate' , 'evaluate')
graph.add_conditional_edges('evaluate',route,{'approved': END, 'need_improvement': 'optimize'})
graph.add_edge('optimize' , 'evaluate')

workflow = graph.compile()

In [27]:
initial_state = {
    "topic": "srhberhb",
    "iteration": 1,
    "max_iteration": 5
}
result = workflow.invoke(initial_state)

GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 14h11m23.997356884s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi